In [ ]:
"""Using a dataset related to the FIFA 2022 video game (included in repository) by Stefano Leone, 
I will load and inspect the data."""

import pandas as pd
import numpy as np                     
import matplotlib.pyplot as plt

# Loading the dataset fifa22.csv and displaying the first 5 rows
data_df = pd.read_csv("fifa22.csv")
data_df
print(data_df.head())

# Calculating the number of observations and features in the dataset
print("Current data observations and features: ", data_df.shape)

In [ ]:
"""Given that the "gender column" provides the gender of the player with binary options of "M" for male 
and "F" for female, I am calculating how many male and female players are in this dataset."""

gender_data = data_df[["gender"]] 

males = 0
females = 0
for index, row in gender_data.iterrows():
    if row["gender"] == "M":
        males +=1
    elif row["gender"] == "F":
        females += 1
        
print("The number of males is", males, " and the number of females is", females)

"""The dataset includes many missing values, particularly for female players- meaning I don't want to drop 
all rows where any value is NaN, as that will disproportionately remove women from the dataset. 
Instead, I want to be more targeted: drop only rows where the column 'passing' contains a missing NaN value. 
The final datadrame should have 17,450 observations."""
data_df
for index, row in data_df.iterrows():
    if str(row["passing"]) == "nan":
        data_df.drop(index, inplace=True)
        
        
print("Changed data observations and features: ", data_df.shape)
    

In [ ]:
"""Here, I will perform multiple linear regression using the statsmodel package."""

import statsmodels.formula.api as smf
import statsmodels.api as sm

"""Using statsmodel package to estimate a multiple regression evaluating the effect on rank of four features: 
passing, attacking, defending, skill."""
results = smf.ols('rank ~ passing + attacking + defending + skill', data=data_df).fit()
results.summary()

In [ ]:
"""Here, I will use SciKit Learn and a test/train split to see how well my model (using the same independent variables
and dependent variable as above) can predict a player's rank."""

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn import metrics
from sklearn.cluster import KMeans
from sklearn import datasets, linear_model

In [ ]:
"""I am creating an X dataframe with just 4 features: passing, attacking, defending, and creating a Y dataframe
with just the 'rank' variable. I will then display the first five rows of each."""
X_dataframe = data_df[['passing', 'attacking', 'defending', 'skill']]
Y_dataframe = data_df[['rank']]
X_dataframe.head()

In [ ]:
# Y dataframe
Y_dataframe.head()

In [ ]:
"""I am creating a test/train split where 25% of the data is held out for testing, using a random seed of 123, and
displaying the first 5 rows of the X training data."""
random = np.random.choice([True, False], size=len(data_df), p=[0.75, 0.25])
train = data_df.loc[random]
test = data_df.loc[random]

X_train = train[['passing', 'attacking', 'defending', 'skill']]
Y_train = train[['rank']]
X_test = test[['passing', 'attacking', 'defending', 'skill']]
Y_test = test[['rank']]

X_train.head()


In [ ]:
"""I am using SKLearn to train a linear regression using only the training data, displaying the intercept
and coefficients for my trained model."""
reg = LinearRegression()
reg.fit(X_train, Y_train)
print("Intercept: ", reg.intercept_)
print("Coefficients: ", reg.coef_)
ypred = reg.predict(X_test)

In [ ]:
"""I am using my trained SKLearn regression model to predict rank values for the hold-out set of X test data, displaying
the first 3 predicred values."""
K_reg = KNeighborsRegressor(n_neighbors=5)
KNeighborsRegressor()
K_reg.fit(X_train, Y_train)
y_pred = K_reg.predict(X_test)
print(y_pred[:3])

In [ ]:
"""I am displaying a scatterplit in which the horizontal axis shows the actual value of the Y test data and the 
vertical axis displays the predicted Y values for the X test data."""
plt.scatter(Y_test, y_pred)
plt.xlabel("Actual value of Y test data")
plt.ylabel("Predicted Y values for X test data")
plt.title("Actual vs. Predicted Y values for the X test data")
plt.show()

In [ ]:
"""I am calculating and displaying the Root Mean Squared Error (RMSE) for this model."""
MSE_reg = np.mean((Y_test - y_pred)**2)
RMSE_reg = MSE_reg**(0.5)
print("RMSE:", RMSE_reg)

In [ ]:
"""Here, I will use KNN to classify players' 'preferred foot.''"""

"""Displaying the count of each value present in the preferred_foot column."""
data_df["preferred_foot"].value_counts()

In [ ]:
"""Calculating which % of players acrually prefer their right foot."""
total_feet = len(data_df["preferred_foot"])
right_foot = data_df["preferred_foot"].value_counts()["Right"]
left_foot = data_df["preferred_foot"].value_counts()["Left"]
probability_rightfoot = (right_foot / total_feet)
probability_leftfoot = (left_foot / total_feet)

print(probability_rightfoot)

In [ ]:
"""Building a classifier using 10 available dimensions: shooting, passing, dribbling, defending, skill, movement,
power, mentality, goalkeeping. Then creating an X dataframe with just these 10 columns and displaying the first
5 rows."""
X_df = data_df[["shooting", "passing", "dribbling", "defending", "attacking", "skill", "movement", "power", "mentality", "goalkeeping"]]
X_df.head()

In [ ]:
"""Rescaling (normalizing) the X data so that each IV has a mean of 0 and a standard deviation of 1, displaying
the first 3 rows of normalized data."""
scaler = StandardScaler()
scaled = scaler.fit_transform(X_df)
print(scaled[:3])

In [ ]:
"""Creating a test-train split to see how well the classifier performs out of sample. I will set Y to be the 
'preferred_foot' column of the database, using 30% of the data for testing anf setting the random state to 456. 
I will display the first 3 rows of X training data."""
X_train, X_test, Y_train, Y_test = train_test_split(scaled, data_df["preferred_foot"], test_size=0.3, random_state=456)
print(X_train[:3])

In [ ]:
"""Calculating the number of neighbors k to consider for my KNN classifier, creating a plot with considered k values
along the horizontal axis and the corresponding error/accuracy displayed along the vertical axis."""
from sklearn.metrics import accuracy_score

k_values = range(1, 31)
accuracy = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, Y_train)
    y_pred = knn.predict(X_test)
    accurate = accuracy_score(Y_test, y_pred)
    accuracy.append(accurate)

plt.plot(k_values, accuracy)
plt.xlabel("k values")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
"""Choosing a reasonable value of k, I am training a KNN classifier that considers this number of neighbors and
predicting Y values (preferred foot) for my out of sample test data, displaying the first 3 predictions for
'preferred foot.'"""
k = 20
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train, Y_train)
y_pref = knn.predict(X_test)
print(y_pref[:3])

In [ ]:
"""Using actual and predicted Y values to calculate and display the confusion matrix for my model, displaying without 
labels but showing classes in alphabetical order (Left, Right; upper left corner is 'Left-Left'). Then, calculating
how many players actually prefer their left food ('True Lefts') were predicted to prefer their right foot."""
from sklearn.metrics import confusion_matrix

y_pred = knn.predict(X_test)
matrix = confusion_matrix(Y_test, y_pred)
print(matrix)
print(matrix[0, 1])

In [ ]:
"""Using the actual and predicted Y values to display the full classification report."""
from sklearn.metrics import classification_report

print(classification_report(Y_test, y_pred))

In [ ]:
"""Here, I will examine natural clusters in the data using K-Means. As the X data, I'll use the same 10 scaled features
that I used for KNN."""

"""Doing some additional pre-processing in order to run the analysis on only a sample of the data, taking the X values
that I calculated previously and converting it to a dataframe, displaying the first 5 rows."""
scaled_df = pd.DataFrame(X_train, columns=X_df.columns)
scaled_df.head()

In [ ]:
"""Randomly sampling the rows of the datadrame, selecting n=5000 rows and setting the rndom state to 2022. I saved
the sampled rows as a new dataframe and displayed the first 5 rows."""
X_train = pd.DataFrame(X_train, columns=X_df.columns)
sample = X_train.sample(n=5000, random_state=2022)
sample.head()

In [ ]:
"""Calculating and saving the error (inertia) and the sillhouette score for possible values of k between 2 and 20
(inclusive). I set the random state to 789 for every value of k considered, and displayed the resulting list of
error values."""
from sklearn.metrics import silhouette_score

inertia = []
silhouette = []
k_values = range(2, 21)
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=789)
    kmeans.fit(sample)
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(sample, kmeans.labels_, random_state=789))

print("Inertia Values: ", inertia)
print("Silhouette Values: ", silhouette)
                   

In [ ]:
"""Plotting the error (inertia) for each value of k."""
plt.plot(k_values, inertia, '-o')
plt.xlabel("Value of K")
plt.ylabel("Inertia")
plt.show()

In [ ]:
"""Using kneed package to estimatr the bend or "elbow" in the curve, then displaying the suggested elbow value."""
import kneed
from kneed import KneeLocator

kneeloc = KneeLocator(k_values, inertia, curve='convex', direction='decreasing')
kneeloc.plot_knee()
print("The suggested elbow value: ", kneeloc.elbow)

In [ ]:
"""Plotting the Silhouette score for each value of k, showing considered values of k along the horizontal axis and 
corresponding Silhouette Scores along the vertical axis."""
plt.plot(k_values, silhouette)
plt.xlabel("Value of K")
plt.ylabel("Silhouette Score")
plt.show()

In [ ]:
"""Based on above analysis, I chose a k value and ran KMeans using the scaled X data, sample to 5k observations and
displayed the first 5 rows of the dataframe."""
kmn = KMeans(n_clusters=10, random_state=789)
kmn.fit(sample)
sample["cluster"] = kmeans.labels_
sample.head()

In [ ]:
"""Creating and displaying a plot which shows a player's attacking scroe on the horizontal axis and their defending score
on the vertical axis."""
plt.scatter(sample["attacking"], sample["defending"], c=kmn.labels_)
plt.xlabel("Attacking Score")
plt.ylabel("Defending Score")
plt.show()